# ESG Data Automation Validation
The core objective of this project is to automate the ingestion, cleaning, and standardization of fragmented, non-standardized facility-level consumption data, compute audited corporate greenhouse gas (GHG) footprints (Scope 1 & Scope 2), and generate financial-grade, structured reporting assets ready for sustainability disclosures.

In [91]:
import pandas as pd

pd.set_option('display.expand_frame_repr', False)

Importing several Excel files (.xlsx) containing data (date, category, resource, quantity and unit of measurement) from different locations.

In [ ]:
data_lione = pd.read_excel(
    'data\consumi_lione.xlsx')
data_milano = pd.read_excel(
    'data\consumi_milano.xlsx')
data_roma = pd.read_excel(
    'data\consumi_roma.xlsx')

Adding the “city” category to identify the source of the data once it has been merged into a single dataframe

In [93]:
data_lione['Città'] = 'Lione'
data_milano['Città'] = 'Milano'
data_roma['Città'] = 'Roma'

print(data_lione)

         Data         Categoria       Risorsa  Quantità  Unità  Città
0  2026-03-01           energia   elettricità    1200.0    KWh  Lione
1  02/03/2026     riscaldamento  gas naturale     250.0    SM3  Lione
2  2026-03-05  flotta aziendale       gasolio       NaN  litri  Lione
3  08/03/2026           energia   elettricità    1300.0    kWh  Lione
4  2026-03-12     riscaldamento  gas naturale     210.0    sm3  Lione


I combine the various dataframes into a single one 

In [94]:
df = pd.concat([data_lione, data_milano, data_roma])
print(df)

         Data             Categoria       Risorsa  Quantità  Unità   Città
0  2026-03-01               energia   elettricità    1200.0    KWh   Lione
1  02/03/2026         riscaldamento  gas naturale     250.0    SM3   Lione
2  2026-03-05      flotta aziendale       gasolio       NaN  litri   Lione
3  08/03/2026               energia   elettricità    1300.0    kWh   Lione
4  2026-03-12         riscaldamento  gas naturale     210.0    sm3   Lione
0  01/03/2026               Energia   Elettricità    1500.5    kWh  Milano
1  2026-03-02         Riscaldamento  Gas Naturale     300.2    sm3  Milano
2         NaN                   NaN           NaN       NaN    NaN  Milano
3  05/03/2026      Flotta aziendale       Gasolio      50.0  litri  Milano
4  2026-03-10               Energia   Elettricità    1450.0    kwh  Milano
0  01/03/2026         Riscaldamento  Gas Naturale     400.0    sm3    Roma
1  03/03/2026               Energia   Elettricità    1800.0    kWh    Roma
2  2026-03-07      Flotta

## Data Cleaning and Standardization
The dataframe contains several rows with anomalies that need to be addressed:
- Missing records
- Inconsistent date and time formats
- Mixed case and non-standard units of measurement for identical metrics

**1.** I will convert the text so that units of measurement are in lower case and categories and resources are capitalised; I also remove any leading or trailing spaces.

In [95]:
df[['Categoria','Risorsa']] = df[['Categoria','Risorsa']].apply(lambda x: x.str.title().str.strip())

df['Unità'] = df['Unità'].str.lower().str.strip()

print(df)

         Data         Categoria       Risorsa  Quantità  Unità   Città
0  2026-03-01           Energia   Elettricità    1200.0    kwh   Lione
1  02/03/2026     Riscaldamento  Gas Naturale     250.0    sm3   Lione
2  2026-03-05  Flotta Aziendale       Gasolio       NaN  litri   Lione
3  08/03/2026           Energia   Elettricità    1300.0    kwh   Lione
4  2026-03-12     Riscaldamento  Gas Naturale     210.0    sm3   Lione
0  01/03/2026           Energia   Elettricità    1500.5    kwh  Milano
1  2026-03-02     Riscaldamento  Gas Naturale     300.2    sm3  Milano
2         NaN               NaN           NaN       NaN    NaN  Milano
3  05/03/2026  Flotta Aziendale       Gasolio      50.0  litri  Milano
4  2026-03-10           Energia   Elettricità    1450.0    kwh  Milano
0  01/03/2026     Riscaldamento  Gas Naturale     400.0    sm3    Roma
1  03/03/2026           Energia   Elettricità    1800.0    kwh    Roma
2  2026-03-07  Flotta Aziendale       Gasolio      80.0  litri    Roma
3     

**2.** I remove the rows containing empty values in the “Quantity” column

In [96]:
df = df.dropna()

print(df)

         Data         Categoria       Risorsa  Quantità  Unità   Città
0  2026-03-01           Energia   Elettricità    1200.0    kwh   Lione
1  02/03/2026     Riscaldamento  Gas Naturale     250.0    sm3   Lione
3  08/03/2026           Energia   Elettricità    1300.0    kwh   Lione
4  2026-03-12     Riscaldamento  Gas Naturale     210.0    sm3   Lione
0  01/03/2026           Energia   Elettricità    1500.5    kwh  Milano
1  2026-03-02     Riscaldamento  Gas Naturale     300.2    sm3  Milano
3  05/03/2026  Flotta Aziendale       Gasolio      50.0  litri  Milano
4  2026-03-10           Energia   Elettricità    1450.0    kwh  Milano
0  01/03/2026     Riscaldamento  Gas Naturale     400.0    sm3    Roma
1  03/03/2026           Energia   Elettricità    1800.0    kwh    Roma
2  2026-03-07  Flotta Aziendale       Gasolio      80.0  litri    Roma
4  15/03/2026           Energia   Elettricità    1600.0    kwh    Roma


**3.** I adjust the date formatting to standardise it to the DD/MM/YYYY format. 

In [97]:
df['Data'] = pd.to_datetime(
    df['Data'], dayfirst=True, format='mixed').dt.strftime('%d/%m/%Y')

print(df)

         Data         Categoria       Risorsa  Quantità  Unità   Città
0  01/03/2026           Energia   Elettricità    1200.0    kwh   Lione
1  02/03/2026     Riscaldamento  Gas Naturale     250.0    sm3   Lione
3  08/03/2026           Energia   Elettricità    1300.0    kwh   Lione
4  12/03/2026     Riscaldamento  Gas Naturale     210.0    sm3   Lione
0  01/03/2026           Energia   Elettricità    1500.5    kwh  Milano
1  02/03/2026     Riscaldamento  Gas Naturale     300.2    sm3  Milano
3  05/03/2026  Flotta Aziendale       Gasolio      50.0  litri  Milano
4  10/03/2026           Energia   Elettricità    1450.0    kwh  Milano
0  01/03/2026     Riscaldamento  Gas Naturale     400.0    sm3    Roma
1  03/03/2026           Energia   Elettricità    1800.0    kwh    Roma
2  07/03/2026  Flotta Aziendale       Gasolio      80.0  litri    Roma
4  15/03/2026           Energia   Elettricità    1600.0    kwh    Roma


**4.** Sort the table by category

In [98]:
df = df.sort_values(by='Categoria')

print(df)

         Data         Categoria       Risorsa  Quantità  Unità   Città
0  01/03/2026           Energia   Elettricità    1200.0    kwh   Lione
3  08/03/2026           Energia   Elettricità    1300.0    kwh   Lione
0  01/03/2026           Energia   Elettricità    1500.5    kwh  Milano
4  10/03/2026           Energia   Elettricità    1450.0    kwh  Milano
1  03/03/2026           Energia   Elettricità    1800.0    kwh    Roma
4  15/03/2026           Energia   Elettricità    1600.0    kwh    Roma
3  05/03/2026  Flotta Aziendale       Gasolio      50.0  litri  Milano
2  07/03/2026  Flotta Aziendale       Gasolio      80.0  litri    Roma
1  02/03/2026     Riscaldamento  Gas Naturale     250.0    sm3   Lione
4  12/03/2026     Riscaldamento  Gas Naturale     210.0    sm3   Lione
1  02/03/2026     Riscaldamento  Gas Naturale     300.2    sm3  Milano
0  01/03/2026     Riscaldamento  Gas Naturale     400.0    sm3    Roma


## Emission Factors
To calculate the emissions associated with the resources used, I use the emission factors for electricity, diesel for the company fleet and natural gas for heating.

### Electricity emission factor
An electricity emission factor (or carbon intensity) measures the greenhouse gas emissions (like CO2) released to generate one kilowatt-hour (kWh) of electricity.
To report on and compare the environmental impact associated with electricity consumption at sites located in Lombardy, Lazio and the Auvergne-Rhône-Alpes region (France), a **location-based approach** applied to final energy consumption was adopted. This methodology quantifies greenhouse gas emissions (expressed in CO2 equivalent) based on the average carbon intensity of the electricity grid to which the sites are connected, calculated not solely on the basis of generation, but on actual meter readings. This means that the factors used incorporate both the direct emissions from local power stations and the impact of energy imported from other areas, including physical transmission and distribution losses across high, medium and low-voltage networks.

For Italy, the emission factors are taken from the ISPRA document (https://www.isprambiente.gov.it/files2025/pubblicazioni/rapporti/r413-2025_def.pdf#page=15.26), which sets out the regional emission factors for 2023:
- Italian Mean **234.7 gCO2/kWh**
- Lombardia **202.2 gCO2/kWh**
- Lazio **291.2 gCO2/kWh**

For France, I am using the national average of **19.6 gCO2/kWh** generated in 2025, as there are no significant regional differences (https://assets.rte-france.com/prod/public/2026-04/Annual-electricity-review-2025-key-findings_0.pdf).


### Diesel emission factor
In addition to electricity consumption, the report includes direct emissions (Scope 1) resulting from the fuel consumption of the company fleet (commercial vehicles and company cars). The calculations focus on the use of diesel, applying the direct fuel combustion approach. To ensure regulatory consistency, the standard emission factor of **2620 gCO2/litre of diesel** consumed has been applied, in accordance with the national ISPRA coefficients (Italy) and the factors from the ADEME Empreinte Database (France). This calculation allows the climate impact of corporate mobility to be quantified regardless of the geographical location of the vehicles, as the calorific value and carbon coefficient of transport fuel are considered to be uniform across Europe.
https://www.fleetup.it/news/calcolo-emissioni-co2-flotta-aziendale/

###  Heating emission factor
The report includes direct emissions (Scope 1) resulting from the combustion of natural gas (methane) used for domestic and industrial heating. To ensure accuracy in line with local billing practices, site consumption has been calculated using the standard Italian national factor of **2.019 gCO2/sm3**
https://www.ets.minambiente.it/Download/237/Tabella%20coefficienti%20standard%20nazionali%202021-2023_v1.pdf

Given the complexity of the emission factors, I am creating a DataFrame to compile them

In [99]:
fattori_emissione = pd.DataFrame([{'Risorsa': 'Elettricità', 'Città': 'Milano', 'Fattore emissione (gCO2)': 202.2},
                                    {'Risorsa': 'Elettricità', 'Città': 'Roma',
                                        'Fattore emissione (gCO2)': 291.2},
                                    {'Risorsa': 'Elettricità', 'Città': 'Lione',
                                        'Fattore emissione (gCO2)': 52},
                                    {'Risorsa': 'Gas Naturale', 'Città': 'Milano',
                                        'Fattore emissione (gCO2)': 2.019},
                                    {'Risorsa': 'Gas Naturale', 'Città': 'Roma',
                                        'Fattore emissione (gCO2)': 2.019},
                                    {'Risorsa': 'Gas Naturale', 'Città': 'Lione',
                                        'Fattore emissione (gCO2)': 2.019},
                                    {'Risorsa': 'Gasolio', 'Città': 'Milano',
                                        'Fattore emissione (gCO2)': 2620},
                                    {'Risorsa': 'Gasolio', 'Città': 'Roma',
                                        'Fattore emissione (gCO2)': 2620},
                                    {'Risorsa': 'Gasolio', 'Città': 'Lione', 'Fattore emissione (gCO2)': 2620}])
print(fattori_conversione)

        Risorsa   Città  Fattore emissione (gCO2)
0   Elettricità  Milano                   202.200
1   Elettricità    Roma                   291.200
2   Elettricità   Lione                    52.000
3  Gas Naturale  Milano                     2.019
4  Gas Naturale    Roma                     2.019
5  Gas Naturale   Lione                     2.019
6       Gasolio  Milano                  2620.000
7       Gasolio    Roma                  2620.000
8       Gasolio   Lione                  2620.000


Now I’ll merge it with the DataFrame containing the resource usage figures.

In [100]:
df = pd.merge(df, fattori_emissione)
print(df)

          Data         Categoria       Risorsa  Quantità  Unità   Città  Fattore emissione (gCO2)
0   01/03/2026           Energia   Elettricità    1200.0    kwh   Lione                    52.000
1   08/03/2026           Energia   Elettricità    1300.0    kwh   Lione                    52.000
2   01/03/2026           Energia   Elettricità    1500.5    kwh  Milano                   202.200
3   10/03/2026           Energia   Elettricità    1450.0    kwh  Milano                   202.200
4   03/03/2026           Energia   Elettricità    1800.0    kwh    Roma                   291.200
5   15/03/2026           Energia   Elettricità    1600.0    kwh    Roma                   291.200
6   05/03/2026  Flotta Aziendale       Gasolio      50.0  litri  Milano                  2620.000
7   07/03/2026  Flotta Aziendale       Gasolio      80.0  litri    Roma                  2620.000
8   02/03/2026     Riscaldamento  Gas Naturale     250.0    sm3   Lione                     2.019
9   12/03/2026     R

## Calculating emissions
Using emission factor data in gCO₂/unit of measurement, I calculate the Scope 1 and Scope 2 emissions resulting from resource consumption.